# HomeRecall on Colab

Index home security recordings with the large models, ask questions, and download the
index to use on your own PC.

Before running: **Runtime > Change runtime type > GPU**. A T4 uses the `colab-t4` profile
(Qwen2.5-VL-3B); an L4 or A100 uses `colab` (Qwen2.5-VL-7B).

Running this means your recordings are uploaded to Google (Drive or this session).

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
PROFILE = "colab" if gb >= 22 else "colab-t4" if gb >= 12 else "laptop"
print(torch.cuda.get_device_name(0), f"{gb:.0f} GB ->", PROFILE)

## 2. Get the code

The repository is private, so the default is to upload its ZIP (GitHub: Code > Download
ZIP). Set `USE_ZIP = False` if you have made it public.

In [ ]:
USE_ZIP = True
REPO_URL = "https://github.com/HameesSajid/homerecall.git"

import glob, os, shutil, zipfile
os.chdir("/content")
if not os.path.isdir("/content/homerecall"):
    if USE_ZIP:
        from google.colab import files
        name = next(iter(files.upload()))
        zipfile.ZipFile(name).extractall("/content/_src")
        root = os.path.dirname(glob.glob("/content/_src/**/recall.py", recursive=True)[0])
        shutil.move(root, "/content/homerecall")
    else:
        !git clone $REPO_URL /content/homerecall
%cd /content/homerecall

## 3. Install the libraries (PyTorch with CUDA is already on Colab)

In [ ]:
!pip -q install -r requirements.txt
!pip -q install https://github.com/ultralytics/CLIP/archive/refs/heads/main.zip

## 4. Point at your recordings

Either put them in a Google Drive folder and set `RECORDINGS` to it, or leave
`USE_DRIVE = False` and upload files directly.

In [ ]:
USE_DRIVE = True
RECORDINGS = "/content/drive/MyDrive/tapo"   # a folder of exported recordings

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    from google.colab import files
    RECORDINGS = "/content/recordings"
    os.makedirs(RECORDINGS, exist_ok=True)
    for name in files.upload():
        shutil.move(name, os.path.join(RECORDINGS, name))
print(len(os.listdir(RECORDINGS)), "files in", RECORDINGS)

## 5. Build the index

The first run downloads the models (about 9 GB for `colab-t4`, 19 GB for `colab`).
Only new recordings are processed when you run it again.

In [ ]:
!python recall.py --profile $PROFILE index "$RECORDINGS"
!python recall.py stats

## 6. What the camera saw

In [ ]:
!python recall.py log

## 7. Ask

`ask()` keeps the models loaded between questions and shows the matching frames.

In [ ]:
import datetime as dt
from homerecall import paths, report
from homerecall.config import pick
from homerecall.search import Searcher
from homerecall.store import Store
from homerecall.models import Models
from IPython.display import Image, display

cfg = pick(PROFILE)
store, models = Store(paths.INDEX), Models(cfg)
searcher = Searcher(store, cfg, detector=models.detector, vlm=models.vlm,
                    embedder=models.embedder if cfg.embedder else None)

def ask(question, verify=None, threshold=0.5, top=5):
    result = searcher.search(question, verify=verify, threshold=threshold)
    if result["matches"]:
        report.save(result, store, report.result_folder(paths.RESULTS, question))
    print(report.format_result(result, top))
    for m in result["matches"][:top]:
        if m.get("sheet"):
            display(Image(m["sheet"]))
    return result

In [ ]:
r = ask("did anyone take eggs out of the fridge")

## 8. Measure it (optional)

Upload a labels file (see the README, "Measuring it on your footage").

In [ ]:
from google.colab import files
labels = next(iter(files.upload()))
!python tools/eval_search.py "$labels" --profile $PROFILE --out outputs/eval_colab.json

## 9. Download the index

Unzip it into the project folder on your PC so that `index/index.db` exists. Colab
deletes the session's files when it disconnects.

In [ ]:
from google.colab import files
shutil.make_archive("/content/index", "zip", "/content/homerecall", "index")
files.download("/content/index.zip")